# `phishing_rules.py` — MODULE 1 : détection d'hameçonnage

**Question centrale : l'identité affichée correspond-elle à l'expéditeur réel ?**

Ce fichier couvre les composantes **1.1 à 1.4** du guide étudiant.
Toutes les fonctions marquées « TODO » sont à compléter par vous.

**CONTRAT IMPOSÉ :** chaque règle renvoie un couple `(points, explication)`.
Un système de sécurité qui affirme sans justifier est inutilisable en SOC.

Pour vérifier votre avancement : `python3 verification.py`

## Imports

- `csv` — pour lire `domaines_officiels.csv`.
- `re` — expressions régulières (recherche de motifs dans du texte).

In [ ]:
import csv
import re

---
## RESSOURCES (fournies)

Données de référence utilisées par les règles :

| Constante | Rôle |
|---|---|
| `MOTS_APPAT` | mots typiques d'un domaine frauduleux (`security`, `login`, `verify`…) |
| `EXTENSIONS_RISQUE` | extensions de premier niveau souvent abusées (`.xyz`, `.tk`…) |
| `SUBSTITUTIONS` | substitutions visuelles employées par les attaquants (`0`→`o`, `rn`→`m`…) |
| `RACCOURCISSEURS` | services de raccourcissement d'URL qui masquent la destination |
| `MOTIF_URL` | expression régulière compilée qui repère `http://` / `https://` |

In [ ]:
MOTS_APPAT = [
    "security", "securite", "login", "verify", "verification",
    "account", "compte", "update", "billing", "suivi",
    "remboursement", "support", "alert", "secure", "signin",
]

EXTENSIONS_RISQUE = [".xyz", ".tk", ".top", ".info", ".gq", ".ml", ".cf"]

SUBSTITUTIONS = {"0": "o", "1": "l", "3": "e", "5": "s", "rn": "m", "vv": "w"}

RACCOURCISSEURS = ["bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd"]

MOTIF_URL = re.compile(r"https?://[^\s<>\"')]+", re.IGNORECASE)

### Fonctions fournies

Ces trois fonctions sont **FOURNIES** : rien à compléter.

#### `charger_domaines_officiels(chemin="domaines_officiels.csv")`

Lit `domaines_officiels.csv` et renvoie un dictionnaire qui associe
chaque marque à la liste de ses domaines légitimes :

```
{"amazon": ["amazon.ca", "amazon.com"], "netflix": ["netflix.com"], ...}
```

Pour chaque ligne du CSV : le nom de la marque est mis en minuscules
(sans espaces autour), et la colonne `domaines_officiels` (des domaines
séparés par des virgules) est découpée en une liste, en ignorant les
cases vides.

#### `tous_les_domaines(table)`

Liste à plat de tous les domaines officiels connus.

#### `extraire_urls(texte)`

Liste des URL présentes dans un texte (via `MOTIF_URL`).

In [ ]:
def charger_domaines_officiels(chemin="domaines_officiels.csv"):
    table = {}

    with open(chemin, encoding="utf-8") as f:
        lecteur = csv.DictReader(f)

        for ligne in lecteur:
            # Nom de la marque : en minuscules, sans espaces autour.
            marque = ligne["marque"].strip().lower()

            # La colonne "domaines_officiels" contient un ou plusieurs
            # domaines separes par des virgules. On les met dans une liste,
            # un par un, en ignorant les cases vides.
            domaines = []
            for domaine in ligne["domaines_officiels"].split(","):
                domaine = domaine.strip().lower()
                if domaine != "":
                    domaines.append(domaine)

            table[marque] = domaines

    return table


def tous_les_domaines(table):
        domaines = []

    for liste_domaines in table.values():
        for domaine in liste_domaines:
            domaines.append(domaine)

    return domaines


def extraire_urls(texte):
   if not texte:
        texte = ""

    urls = MOTIF_URL.findall(texte)

    return urls

---
## COMPOSANTE 1.1 — CLASSIFICATION DES DOMAINES

### TODO — `extraire_domaine_adresse(adresse)`

Domaine d'une adresse courriel, en minuscules.

| Entrée | Sortie |
|---|---|
| `"Amazon Support <support123@gmail.com>"` | `"gmail.com"` |
| `"support123@gmail.com"` | `"gmail.com"` |
| `""` | `""` |

**Démarche :** couper au `"@"`, garder la partie de droite, retirer un `">"` final.

In [ ]:
def extraire_domaine_adresse(adresse):
    pass

### TODO — `extraire_domaine_url(url)`

Domaine d'une URL, en minuscules.

| Entrée | Sortie |
|---|---|
| `"https://www.amazon.ca/gp/orders"` | `"www.amazon.ca"` |
| `"http://amazon.ca.security-check.xyz/login"` | `"amazon.ca.security-check.xyz"` |

**Démarche :** retirer `"http://"` ou `"https://"`, couper au premier `"/"`,
puis retirer un éventuel `":port"`.

In [ ]:
def extraire_domaine_url(url):
    pass

### TODO — `domaine_est_officiel(domaine, domaines_officiels)`

`True` si `domaine` est un domaine officiel **OU** un sous-domaine
d'un domaine officiel.

| Entrée | Sortie |
|---|---|
| `"accesd.desjardins.com"` | `True` |
| `"desjardins.com.securite-client.xyz"` | `False` ← **LE PIÈGE** |

**ATTENTION :** comparez la **FIN** du domaine, jamais son contenu.
Un test `if "desjardins.com" in domaine` classe le second comme
légitime : c'est exactement l'erreur que l'attaquant provoque.

> **Indice :** `domaine == officiel or domaine.endswith("." + officiel)`

In [ ]:
def domaine_est_officiel(domaine, domaines_officiels):
    pass

### TODO — `regle_mot_appat(domaine)`

**+25** si le domaine contient un mot de `MOTS_APPAT`.
Renvoie `(points, explication)`. Sinon : `(0, "")`.

In [ ]:
def regle_mot_appat(domaine):
    pass

### TODO — `regle_extension_risque(domaine)`

**+10** si le domaine se termine par une extension de
`EXTENSIONS_RISQUE`. Renvoie `(points, explication)`.

In [ ]:
def regle_extension_risque(domaine):
    pass

### TODO — `regle_substitution(domaine, marques_connues)`

**+30** si, après avoir annulé les substitutions de `SUBSTITUTIONS`,
le domaine contient le nom d'une marque connue **ALORS QUE** le domaine
d'origine ne le contenait pas.

| Domaine | Après annulation | Résultat |
|---|---|---|
| `"amaz0n.com"` | `"amazon.com"` | marque « amazon » → **+30** |
| `"rnicrosoft.com"` | `"microsoft.com"` | marque « microsoft » → **+30** |
| `"amazon.ca"` | inchangé | rien → **0** |

`marques_connues` est la liste des clés de la table (`["amazon", ...]`).

In [ ]:
def regle_substitution(domaine, marques_connues):
    pass

---
## COMPOSANTE 1.2 — ANALYSE DES LIENS

### TODO — `marque_annoncee(courriel, marques_connues)`

Renvoie la marque que le courriel **PRÉTEND** représenter, ou `None`.
On cherche dans le nom affiché, le sujet et le message.

| Entrée | Sortie |
|---|---|
| `name="Netflix", subject="Echec de paiement"` | `"netflix"` |
| `name="Super Rabais", subject="LIQUIDATION"` | `None` |

Pourquoi cette fonction est indispensable : voir `regle_lien_incoherent()`.

Renvoie la marque que le courriel PRÉTEND représenter, ou None.

    On cherche dans le nom affiché, le sujet et le message.

La fonction en résumé : 
courriel
   ↓
récupérer name
   ↓
récupérer subject
   ↓
récupérer message
   ↓
mettre tout dans texte
   ↓
mettre en minuscules
   ↓
parcourir marques_connues
   ↓
chercher chaque marque dans texte
   ↓
marque trouvée → return marque
   ↓
aucune marque → return None

In [ ]:
def marque_annoncee(courriel, marques_connues):
    pass

### TODO — `regle_lien_incoherent(url, marque, table_marques)`

**+25** si le courriel se réclame d'une marque (`marque` non `None`)
**ALORS QUE** le lien pointe ailleurs que vers un domaine officiel de
cette marque.

Si `marque` vaut `None` : renvoyer `(0, "")` — la règle ne s'applique pas.

#### POURQUOI CETTE CONDITION EST OBLIGATOIRE

Une première version naïve dirait : « +25 pour tout lien vers un
domaine non officiel ». Mais la quasi-totalité des liens du monde
pointent vers des domaines absents de votre liste blanche : le site
d'un fournisseur, un article de presse, une boutique locale.
Cette règle-là signalerait presque **TOUS** les courriels, y compris les
légitimes — et un détecteur qui crie tout le temps ne détecte rien.

L'hameçonnage n'est pas « un lien inconnu ». C'est un lien inconnu
**QUI SE FAIT PASSER POUR** une marque connue. C'est la contradiction
qui est l'indice, pas le lien lui-même.

In [ ]:
def regle_lien_incoherent(url, marque, table_marques):
    pass

### TODO — `regle_lien_adresse_ip(url)`

**+25** si le domaine du lien est une adresse IP.

    "http://198.51.100.77/login"  ->  (25, "...")

> **Indice :** `re.fullmatch(r"\d+\.\d+\.\d+\.\d+", domaine)`

Une vraie organisation possède un nom de domaine. Une adresse IP nue
    signale une machine compromise ou un hébergement jetable — et elle
    empêche la victime de reconnaître un nom familier.

In [ ]:
def regle_lien_adresse_ip(url):
    pass

### TODO — `regle_lien_raccourcisseur(url)`

**+15** si le domaine figure dans `RACCOURCISSEURS`.

 Un raccourcisseur cache la destination réelle : impossible de juger
    le lien avant de cliquer. Ce n'est pas une preuve de fraude. Les infolettres légitimes en utilisent, d'où un score modéré.

In [ ]:
def regle_lien_raccourcisseur(url):
    pass

### TODO — `regle_lien_sans_https(url)`

 +10 si l'URL ne commence pas par "https://".

    Signal faible en 2026 : le chiffrement est devenu la norme, et un
    site d'hameçonnage peut très bien avoir un certificat. Score modéré
    en conséquence.

In [ ]:
def regle_lien_sans_https(url):
    pass

---
## COMPOSANTE 1.3 — NOM AFFICHÉ vs ADRESSE D'ENVOI

### TODO — `marque_dans_nom(nom_affiche, marques_connues)`

Renvoie le nom de la marque détectée dans le nom affiché,
ou `None` si aucune.

| Entrée | Sortie |
|---|---|
| `"Amazon Support"` | `"amazon"` |
| `"Marie Tremblay"` | `None` |

Rappel : le nom affiché est choisi LIBREMENT par l'expéditeur.
    N'importe qui peut s'appeler « Amazon Support ». C'est précisément
    pour cela qu'il faut le confronter à l'adresse réelle.

In [ ]:
def marque_dans_nom(nom_affiche, marques_connues):
    pass

### TODO — `regle_nom_vs_adresse(nom_affiche, adresse, table_marques)`

**+20** si le nom affiché évoque une marque connue **ALORS QUE** le
domaine de l'adresse n'est pas un domaine officiel de **CETTE** marque.

Le raisonnement se fait en deux temps :

1. une marque est-elle citée dans le nom affiché ?
   si non → renvoyer `(0, "")` et ne rien signaler
2. si oui → le domaine de l'adresse est-il officiel **POUR ELLE** ?

Une règle qui se déclenche sur tous les courriels ne discrimine rien.

In [ ]:
def regle_nom_vs_adresse(nom_affiche, adresse, table_marques):
    pass

---
## COMPOSANTE 1.4 — EN-TÊTES TECHNIQUES

### TODO — `regle_return_path(entete_from, entete_return_path)`

**+15** si le domaine de `From` diffère de celui de `Return-Path`.

Si `Return-Path` est vide ou absent : renvoyer `(0, "")`.
Une donnée manquante n'est pas une preuve de fraude.

**PIÈGE :** `"info@infolettre-cuisine.com"` et
`"bounce-4471@infolettre-cuisine.com"` ont le **MÊME** domaine.
C'est le fonctionnement normal d'une infolettre.
Comparez les domaines, jamais les adresses complètes.

In [ ]:
def regle_return_path(entete_from, entete_return_path):
    pass

### TODO — `regle_reply_to(entete_from, entete_reply_to)`

**+15** si `Reply-To` est présent **ET** si son domaine diffère de
celui de `From`. Si `Reply-To` est vide : renvoyer `(0, "")`.

In [ ]:
def regle_reply_to(entete_from, entete_reply_to):
    pass

---
## ASSEMBLAGE (fourni — ne pas modifier)

`analyser_phishing(courriel, table_marques)` applique toutes les règles du
module 1 à un courriel.

`courriel` est un dictionnaire avec les clés :
`sender`, `name`, `subject`, `message`, `return_path`, `reply_to`.

Renvoie `(score_brut, [(points, explication), ...])`.

**Structure de l'analyse :**
- Composante 1.1 — le domaine de l'expéditeur (uniquement s'il n'est pas officiel)
- Composante 1.2 — chaque lien du message
- Composante 1.3 — nom affiché vs adresse
- Composante 1.4 — en-têtes `Return-Path` et `Reply-To`

In [ ]:
def _ajouter(resultats, couple):
    """Ajoute (points, explication) au bilan, en ignorant les None."""
    if not couple:
        return
    points, explication = couple
    if points:
        resultats.append((points, explication))


def analyser_phishing(courriel, table_marques):
    officiels = tous_les_domaines(table_marques)
    marques = list(table_marques.keys())
    details = []

    domaine_exp = extraire_domaine_adresse(courriel.get("sender", "")) or ""

    # Composante 1.1 — le domaine de l'expéditeur
    if domaine_exp and not domaine_est_officiel(domaine_exp, officiels):
        _ajouter(details, regle_mot_appat(domaine_exp))
        _ajouter(details, regle_extension_risque(domaine_exp))
        _ajouter(details, regle_substitution(domaine_exp, marques))

    # Composante 1.2 — les liens du message
    texte = f"{courriel.get('subject', '')} {courriel.get('message', '')}"
    marque = marque_annoncee(courriel, marques)
    for url in extraire_urls(texte):
        _ajouter(details, regle_lien_incoherent(url, marque, table_marques))
        _ajouter(details, regle_lien_adresse_ip(url))
        _ajouter(details, regle_lien_raccourcisseur(url))
        _ajouter(details, regle_lien_sans_https(url))

    # Composante 1.3 — nom affiché vs adresse
    _ajouter(details, regle_nom_vs_adresse(
        courriel.get("name", ""), courriel.get("sender", ""), table_marques))

    # Composante 1.4 — en-têtes
    _ajouter(details, regle_return_path(
        courriel.get("sender", ""), courriel.get("return_path", "")))
    _ajouter(details, regle_reply_to(
        courriel.get("sender", ""), courriel.get("reply_to", "")))

    return sum(p for p, _ in details), details